In [31]:
url = 'https://data.gov.il/api/3/action/datastore_search?resource_id=053cea08-09bc-40ec-8f7a-156f0677aff3'

import requests
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import widgets, VBox, Output
from IPython.display import display

pd.set_option("display.max_columns", None)

response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)

data_df = pd.DataFrame(data['result']['records'])

# Tab 1: Data Overview
tab1_content = widgets.Output()
with tab1_content:
    print("Data Overview:")
    display(data_df.describe())  # Summary statistics

# Tab 2: Raw Data
tab2_content = widgets.Output()
with tab2_content:
    print("Raw Data:")
    display(data_df)

# Tab 3: Chart
tab3_content = widgets.Output()
with tab3_content:
    print("Charts:")
    plt.clf()
    plt.figure(figsize=(12, 6))

    # Data processing for the graph
    # Drop rows where 'shnat_yitzur' is missing for accurate year-based counting
    cars_per_year_raw = data_df.dropna(subset=['shnat_yitzur']).copy()
    cars_per_year_raw['shnat_yitzur'] = pd.to_numeric(cars_per_year_raw['shnat_yitzur'], errors='coerce')
    cars_per_year_raw.dropna(subset=['shnat_yitzur'], inplace=True)
    cars_per_year_raw['shnat_yitzur'] = cars_per_year_raw['shnat_yitzur'].astype(int)

    # Group by production year and count the total number of cars (rows)
    total_cars_count = cars_per_year_raw.groupby('shnat_yitzur').size().reset_index(name='total_cars')
    total_cars_count = total_cars_count.sort_values('shnat_yitzur')

    # Create the bar chart
    plt.bar(total_cars_count['shnat_yitzur'], total_cars_count['total_cars'], color='skyblue')
    plt.xlabel('Production Year')
    plt.ylabel('Total Number of Cars')
    plt.title('Total Number of Cars Produced Per Year')
    plt.xticks(rotation=90)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

tabs = widgets.Tab(children=[tab1_content, tab2_content,tab3_content])
tabs.set_title(0, 'Tab 1: Data Overview')
tabs.set_title(1, 'Tab 2: Raw Data')
tabs.set_title(2, 'Tab 3: Charts')

# Display Tabs
display(tabs)